In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")

df.columns = df.columns.astype(str).str.strip()
df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]

target_col = None

for col in df.columns:
    if col.lower().replace("_", "").replace(" ", "") in [
        "medhouseval", "medianhousevalue", "target"
    ]:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found. Check dataset columns.")

X = df.drop(columns=[target_col]).select_dtypes(include="number")
y = pd.to_numeric(df[target_col], errors="coerce")

valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model1 = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Lasso(alpha=0.1, max_iter=10000))
])

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)

pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)

train_pred1 = model1.predict(X_train)
train_pred2 = model2.predict(X_train)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Lasso Regression"],
    "Training MSE": [
        mean_squared_error(y_train, train_pred1),
        mean_squared_error(y_train, train_pred2)
    ],
    "Testing MSE": [
        mean_squared_error(y_test, pred1),
        mean_squared_error(y_test, pred2)
    ],
    "Training R2": [
        r2_score(y_train, train_pred1),
        r2_score(y_train, train_pred2)
    ],
    "Testing R2": [
        r2_score(y_test, pred1),
        r2_score(y_test, pred2)
    ]
})

print("Model Performance Comparison:")
print(results.to_string(index=False))

lasso_coef = model2.named_steps["regressor"].coef_

feature_comparison = pd.DataFrame({
    "Feature": X.columns,
    "Lasso Coefficient": lasso_coef
})

print("\nLasso Feature Coefficients:")
print(feature_comparison.to_string(index=False))

selected_features = X.columns[np.abs(lasso_coef) > 1e-8]

print("\nSelected Features:")
print(list(selected_features))

print("\nTotal Original Features:", X.shape[1])
print("Selected Features:", len(selected_features))
print("Removed Features:", X.shape[1] - len(selected_features))


Model Performance Comparison:
            Model  Training MSE  Testing MSE  Training R2  Testing R2
Linear Regression      0.517933     0.555892     0.612551    0.575788
 Lasso Regression      0.671840     0.679629     0.497419    0.481361

Lasso Feature Coefficients:
   Feature  Lasso Coefficient
    MedInc           0.710598
  HouseAge           0.106453
  AveRooms          -0.000000
 AveBedrms           0.000000
Population          -0.000000
  AveOccup          -0.000000
  Latitude          -0.011469
 Longitude          -0.000000

Selected Features:
['MedInc', 'HouseAge', 'Latitude']

Total Original Features: 8
Selected Features: 3
Removed Features: 5


In [ ]:
%pip install pandas numpy scikit-learn
# 1. AIM

To implement Lasso Regression on a suitable multivariate open-access dataset and analyze its effect on model overfitting and feature selection.

# 2. DATASET USED

**Dataset Name:** California Housing Dataset

**Dataset Path:** `C:\Users\Tanuja\Downloads\california_housing.csv`

The California Housing dataset contains housing-related and demographic information about different districts in California.

It is used to predict median house value based on multiple independent variables.

The standard dataset contains 20,640 records, 8 numerical input features and one continuous target variable.

### Features of the Dataset

| Feature | Description |
|---|---|
| MedInc | Median income in the block group |
| HouseAge | Median house age |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Population of the block group |
| AveOccup | Average number of household members |
| Latitude | Geographical latitude |
| Longitude | Geographical longitude |
| MedHouseVal | Median house value (target) |

**Why is this dataset used?**

1. It contains multiple numerical independent variables.
2. It represents a real-world housing prediction problem.
3. It is suitable for applying Linear Regression and Lasso Regression.
4. It helps demonstrate the effect of L1 regularization.
5. It allows identification of less useful features through coefficient reduction.
6. It helps observe the effect of regularization on model overfitting.

# 3. THEORY

## A. Regularization

Regularization is a technique used in machine learning to reduce overfitting by adding a penalty term to the model's loss function.

It controls model complexity by restricting the magnitude of regression coefficients.

Regularization helps the model generalize better to unseen data.

**Need for Regularization:**

1. Reduces the possibility of overfitting.
2. Controls large regression coefficients.
3. Helps improve model stability.
4. Performs feature selection in certain regularization techniques.
5. Improves generalization in suitable situations.

## B. Lasso Regression

Lasso stands for **Least Absolute Shrinkage and Selection Operator**.

Lasso Regression is a regularized version of Linear Regression that uses L1 regularization.

It adds a penalty equal to the sum of the absolute values of regression coefficients to the loss function.

Lasso Regression reduces coefficient magnitudes and can make some coefficients exactly zero.

Therefore, it performs automatic feature selection by eliminating features whose coefficients become zero.

### Mathematical Formula

Lasso Regression minimizes:

L = Σ(Yᵢ - Ŷᵢ)² + αΣ|βⱼ|

Where:

- Yᵢ = Actual value.
- Ŷᵢ = Predicted value.
- βⱼ = Regression coefficients.
- α = Regularization parameter.
- Σ(Yᵢ - Ŷᵢ)² = Sum of Squared Errors.
- αΣ|βⱼ| = L1 regularization penalty.

The first term minimizes prediction errors, while the second term controls the magnitude of regression coefficients.

## C. L1 Regularization

L1 Regularization adds the absolute values of regression coefficients as a penalty to the loss function.

It reduces coefficient magnitudes and can make some coefficients exactly zero.

When a coefficient becomes zero, its corresponding feature is excluded from the model's predictions.

Lasso Regression uses L1 regularization.

## D. Alpha Parameter in Lasso Regression

Alpha (α) controls the strength of the regularization penalty.

- **Alpha = 0:** Corresponds to ordinary least squares without regularization.
- **Small Alpha:** Applies a weaker penalty.
- **Large Alpha:** Applies a stronger penalty and increases coefficient shrinkage.
- **Very Large Alpha:** May make many coefficients zero and cause underfitting.

In this practical, alpha = 0.1 is used.

It helps demonstrate how L1 regularization affects regression coefficients and feature selection.

## E. Feature Selection Using Lasso

Feature Selection is the process of selecting important input features and removing unnecessary features from a dataset.

Lasso performs automatic feature selection by shrinking some regression coefficients exactly to zero.

Example:

| Feature | Lasso Coefficient | Result |
|---|---:|---|
| Feature A | 0.85 | Selected |
| Feature B | 0.00 | Removed |
| Feature C | -0.42 | Selected |
| Feature D | 0.00 | Removed |

In this example, Feature A and Feature C are selected because their coefficients are non-zero.

Feature B and Feature D are excluded because their coefficients are zero.

In this practical, feature selection is observed by checking the Lasso regression coefficients.

## F. Difference Between Linear Regression and Lasso Regression

**Linear Regression:**

Linear Regression uses the Ordinary Least Squares method to minimize the sum of squared prediction errors without adding a regularization penalty.

**Lasso Regression:**

Lasso Regression adds an L1 penalty to the loss function. It shrinks coefficients and can eliminate some features by making their coefficients exactly zero.

| Linear Regression | Lasso Regression |
|---|---|
| No regularization penalty | Uses L1 regularization |
| Uses squared error minimization | Uses squared error with an L1 penalty |
| Does not perform automatic feature selection | Can perform automatic feature selection |
| Coefficients may become large | Coefficients are controlled |
| May overfit in some situations | Helps control overfitting |

## G. Overfitting

Overfitting occurs when a machine learning model learns the training data too closely, including noise and unnecessary patterns.

An overfitted model performs well on training data but poorly on unseen testing data.

**Characteristics of Overfitting:**

1. Very low training error.
2. Higher testing error.
3. Poor generalization.
4. High model variance.

Lasso Regression helps reduce overfitting by penalizing large regression coefficients and reducing unnecessary model complexity.

However, it does not guarantee that overfitting will always be eliminated.

## H. Underfitting

Underfitting occurs when a model is too simple to learn the important relationships in the data.

In Lasso Regression, an excessively high alpha value may shrink too many coefficients towards zero, causing underfitting.

**Characteristics:**

1. High training error.
2. High testing error.
3. Poor learning of important patterns.
4. High bias.

## I. Feature Scaling

Feature Scaling is the process of transforming numerical attributes into a comparable scale.

It is important in Lasso Regression because the L1 penalty acts on coefficient magnitudes. Features having different numerical scales may otherwise receive unequal effective regularization.

In this practical, StandardScaler is used before applying Lasso.

### StandardScaler Formula

Z = (X - μ) / σ

Where:

- X = Original value.
- μ = Mean of the feature.
- σ = Standard deviation.
- Z = Standardized value.

## J. Evaluation of Lasso Regression

In this practical, Linear Regression and Lasso Regression are compared using the following metrics.

### 1. Mean Squared Error (MSE)

MSE calculates the average squared difference between actual and predicted values.

Formula:

MSE = (1/n) × Σ(Yᵢ - Ŷᵢ)²

A lower MSE indicates smaller squared prediction errors.

### 2. R² Score

R² measures the proportion of variation in the dependent variable explained by the regression model.

Formula:

R² = 1 - (SSres / SStot)

A higher R² generally indicates better model performance. It can also be negative if the model performs worse than a constant mean-based prediction.

### 3. Regression Coefficients

Regression coefficients represent the estimated contribution of input features to the predicted output.

In this practical, Lasso coefficients are examined to identify which features are retained and which are eliminated.

# 4. ALGORITHM

1. Start.
2. Import Pandas, NumPy and required Scikit-learn libraries.
3. Load the California Housing dataset using read_csv().
4. Clean the column names and remove unwanted unnamed columns.
5. Identify the target variable.
6. Separate independent variables X and dependent variable y.
7. Remove rows containing missing input or target values.
8. Divide the dataset into training and testing sets.
9. Create a Linear Regression model.
10. Standardize the input features using StandardScaler.
11. Train the Linear Regression model.
12. Create a Lasso Regression model with alpha = 0.1.
13. Standardize the input features and train the Lasso Regression model.
14. Predict output values using both models.
15. Calculate training and testing MSE.
16. Calculate training and testing R² scores.
17. Extract the Lasso regression coefficients.
18. Identify features having non-zero coefficients.
19. Count the selected and removed features.
20. Display the comparison results.
21. Stop.

# 5. DETAILED EXPLANATION OF EACH LINE OF CODE

## A. Importing Libraries

**1. import pandas as pd**

Imports Pandas for loading and manipulating datasets.

**2. import numpy as np**

Imports NumPy for numerical calculations and checking coefficient values.

**3. from sklearn.model_selection import train_test_split**

Imports train_test_split() to divide the dataset into training and testing sets.

**4. from sklearn.preprocessing import StandardScaler**

Imports StandardScaler to standardize numerical features before regression.

**5. from sklearn.pipeline import Pipeline**

Imports Pipeline to combine feature scaling and regression into one workflow.

**6. from sklearn.linear_model import LinearRegression, Lasso**

Imports two regression algorithms:

- LinearRegression implements ordinary least squares regression.
- Lasso implements regression with L1 regularization.

**7. from sklearn.metrics import mean_squared_error, r2_score**

Imports two evaluation functions:

- mean_squared_error() calculates MSE.
- r2_score() calculates R².

## B. Loading the Dataset

`df = pd.read_csv(r"C:\Users\Tanuja\Downloads\california_housing.csv")`

- read_csv() reads the CSV file.
- df stores the dataset in a Pandas DataFrame.
- The specified path identifies the dataset location.

## C. Cleaning Column Names

`df.columns = df.columns.astype(str).str.strip()`

Converts column names into strings and removes extra spaces.

`df = df.loc[:, ~df.columns.str.lower().str.startswith("unnamed")]`

Removes unwanted unnamed columns that may have been generated during CSV export.

## D. Identifying the Target Variable

`target_names = ["medhouseval", "medianhousevalue", "target"]`

Creates a list containing common names of the target variable.

`target_col = next(...)`

Searches the dataset and identifies the target column.

`if target_col is None:`

Checks whether a target column has been found.

`raise ValueError(...)`

Displays an error if the target column cannot be identified.

## E. Selecting Independent Variables

`X = df.drop(columns=[target_col]).select_dtypes(include="number")`

- drop() removes the target column.
- select_dtypes(include="number") selects numerical input features.
- X stores the independent variables.

## F. Selecting Dependent Variable

`y = pd.to_numeric(df[target_col], errors="coerce")`

Converts the target column into numerical values.

`errors="coerce"` converts invalid entries into missing values.

The variable y stores the dependent variable.

## G. Removing Missing Values

`valid_rows = X.notna().all(axis=1) & y.notna()`

Checks whether all input features and the target value are available in each row.

`X = X.loc[valid_rows]`

Keeps valid input records.

`y = y.loc[valid_rows]`

Keeps their corresponding target values.

## H. Splitting the Dataset

`train_test_split(X, y, test_size=0.2, random_state=42)`

Divides the dataset into:

- 80% training data.
- 20% testing data.

random_state=42 ensures reproducible splitting.

## I. Creating Linear Regression Model

`model1 = Pipeline([...])`

Creates a pipeline containing:

1. StandardScaler.
2. LinearRegression.

StandardScaler standardizes the features, and LinearRegression creates the baseline model.

## J. Creating Lasso Regression Model

`model2 = Pipeline([...])`

Creates a pipeline containing:

1. StandardScaler.
2. Lasso Regression.

`Lasso(alpha=0.1, max_iter=10000)`

Creates a Lasso Regression model.

- alpha=0.1 specifies the L1 regularization strength.
- max_iter=10000 specifies the maximum number of iterations allowed for optimization.

## K. Training Both Models

`model1.fit(X_train, y_train)`

Trains the Linear Regression model using the training dataset.

`model2.fit(X_train, y_train)`

Trains the Lasso Regression model using the same training dataset.

The scaling parameters are learned from training data only.

## L. Making Predictions

`pred1 = model1.predict(X_test)`

Predicts house values using Linear Regression.

`pred2 = model2.predict(X_test)`

Predicts house values using Lasso Regression.

`train_pred1 = model1.predict(X_train)`

Predicts training values using Linear Regression.

`train_pred2 = model2.predict(X_train)`

Predicts training values using Lasso Regression.

## M. Comparing Model Performance

`mean_squared_error(y_train, train_pred1)`

Calculates training MSE for Linear Regression.

`mean_squared_error(y_train, train_pred2)`

Calculates training MSE for Lasso Regression.

`mean_squared_error(y_test, pred1)`

Calculates testing MSE for Linear Regression.

`mean_squared_error(y_test, pred2)`

Calculates testing MSE for Lasso Regression.

Similarly, r2_score() calculates training and testing R² scores for both models.

## N. Extracting Lasso Coefficients

`lasso_coef = model2.named_steps["regressor"].coef_`

Extracts the regression coefficients learned by the Lasso model.

These coefficients are used to identify which features are retained and which are eliminated.

## O. Identifying Selected Features

`np.abs(lasso_coef) > 1e-8`

Checks which coefficients have a magnitude greater than a very small threshold.

`X.columns[np.abs(lasso_coef) > 1e-8]`

Returns the names of features having non-zero coefficients.

These features are considered selected by Lasso.

## P. Counting Selected and Removed Features

`len(selected_features)`

Calculates the number of selected features.

`X.shape[1]`

Returns the total number of original input features.

`X.shape[1] - len(selected_features)`

Calculates the number of features eliminated by Lasso.

# 6. EXPECTED OUTPUT

For the standard California Housing dataset:

- Total records: 20,640.
- Original input features: 8.
- Target variable: MedHouseVal.
- Training records: 16,512.
- Testing records: 4,128.
- Lasso regularization parameter: 0.1.

The program displays:

1. Model Performance Comparison.
2. Training MSE of both models.
3. Testing MSE of both models.
4. Training R² of both models.
5. Testing R² of both models.
6. Lasso Feature Coefficients.
7. Selected Features.
8. Total Original Features.
9. Number of Selected Features.
10. Number of Removed Features.

The actual coefficients and selected features depend on the dataset and chosen alpha value. They should be taken from your executed output.

# 7. ORAL EXAMINATION – VIVA QUESTIONS AND ANSWERS

## Basic Questions

**Q1. What is Regularization?**

Regularization is a technique used to reduce overfitting by adding a penalty term to the model's loss function.

**Q2. What is Lasso Regression?**

Lasso Regression is a regularized version of Linear Regression that uses L1 regularization.

**Q3. What does LASSO stand for?**

Least Absolute Shrinkage and Selection Operator.

**Q4. What is L1 Regularization?**

L1 Regularization adds the absolute values of regression coefficients as a penalty to the loss function.

**Q5. Why is Lasso Regression used?**

It helps reduce overfitting and performs automatic feature selection by making some coefficients exactly zero.

**Q6. What is Alpha in Lasso Regression?**

Alpha is the regularization parameter that controls the strength of the L1 penalty.

**Q7. What happens when Alpha increases?**

The regularization penalty becomes stronger, causing more coefficients to shrink and potentially become zero.

**Q8. What happens when Alpha is very small?**

The penalty is weak, so the model behaves more like ordinary Linear Regression.

**Q9. What happens when Alpha is very large?**

Many coefficients may become zero, potentially causing underfitting.

**Q10. What is a regression coefficient?**

It represents the estimated change in the predicted output for a one-unit change in an input feature, keeping other features constant.

## Feature Selection Questions

**Q11. What is Feature Selection?**

Feature Selection is the process of selecting important input features and excluding unnecessary features.

**Q12. How does Lasso perform Feature Selection?**

It shrinks some regression coefficients exactly to zero. The features corresponding to those zero coefficients are eliminated from the model.

**Q13. Does Lasso remove the original columns from the dataset?**

No. It gives zero coefficients to eliminated features, so those features do not contribute to the fitted model's predictions.

**Q14. What is the difference between Ridge and Lasso Regression?**

Ridge uses L2 regularization and generally shrinks coefficients towards zero. Lasso uses L1 regularization and can make some coefficients exactly zero.

**Q15. Can Lasso select all features?**

Yes. If the regularization strength is sufficiently weak, all features may retain non-zero coefficients.

# 8. DIFFICULT EXTERNAL EXAMINER QUESTIONS AND ANSWERS

**Q16. What is the mathematical objective of Lasso Regression?**

It minimizes the sum of squared prediction errors along with an L1 penalty on the regression coefficients.

**Q17. Why is Lasso called a feature selection technique?**

Because it can make some regression coefficients exactly zero, automatically eliminating the corresponding features from the fitted model.

**Q18. Does Lasso always improve testing accuracy?**

No. Its performance depends on the dataset and the chosen alpha value. Excessive regularization can cause underfitting.

**Q19. What is the difference between L1 and L2 regularization?**

L1 uses absolute coefficient values as a penalty, whereas L2 uses squared coefficient values.

**Q20. Can Lasso coefficients become exactly zero?**

Yes. L1 regularization encourages sparse coefficient solutions, so some coefficients can become exactly zero.

**Q21. Why is StandardScaler applied before Lasso?**

Because Lasso penalizes coefficient magnitudes. Scaling ensures that input features with different numerical scales are treated more comparably by the regularization penalty.

**Q22. What is the difference between overfitting and underfitting?**

Overfitting occurs when a model learns the training data too closely, whereas underfitting occurs when the model is too simple to learn important patterns.

**Q23. How does Lasso reduce overfitting?**

It controls model complexity by penalizing large coefficients and eliminating some less useful features.

**Q24. What happens when Alpha is zero?**

There is no L1 penalty, so the objective corresponds to ordinary least squares. In practice, a regularized Lasso model should use a positive alpha value.

**Q25. What is the difference between Ridge and Lasso in feature selection?**

Ridge generally retains all input features with reduced coefficients, whereas Lasso can eliminate some features by making their coefficients zero.

**Q26. What happens when two features are highly correlated?**

Lasso may select one feature, both features or a subset of correlated features. Its selection can be unstable when predictors are strongly correlated.

**Q27. What is Bias–Variance Trade-off?**

It is the balance between bias and variance to achieve good model generalization.

**Q28. Can Lasso be used when the number of features is greater than the number of observations?**

Yes. Lasso can be useful in high-dimensional problems because it encourages sparse solutions.

**Q29. Why is max_iter=10000 used?**

It specifies the maximum number of optimization iterations, giving the model more opportunities to converge.

**Q30. Why do we compare training and testing errors?**

To observe whether regularization improves generalization and controls overfitting.

**Q31. What is the limitation of Lasso when features are highly correlated?**

It may select one correlated feature arbitrarily and remove another, making feature selection unstable.

**Q32. How can we select the best alpha value?**

We can test different alpha values using cross-validation and select the value that provides the best validation performance.

**Q33. Does Lasso perform dimensionality reduction?**

Lasso performs feature selection. It reduces the number of features used by the model when some coefficients become zero.

**Q34. What is the difference between Lasso and Elastic Net?**

Lasso uses L1 regularization, whereas Elastic Net combines L1 and L2 regularization.

**Q35. Does Lasso completely remove multicollinearity?**

No. Lasso may remove some correlated features, but it does not guarantee that multicollinearity is completely eliminated.

# 9. DATASET-RELATED QUESTIONS

**Q1. What is the California Housing dataset?**

It is a real-world dataset containing housing and demographic information about California districts.

**Q2. How many records are present in the standard dataset?**

20,640 records.

**Q3. How many numerical input features are present?**

Eight numerical features.

**Q4. What is the target variable?**

MedHouseVal, which represents median house value.

**Q5. What does MedInc represent?**

Median income in the block group.

**Q6. What does HouseAge represent?**

Median house age in the block group.

**Q7. What does AveRooms represent?**

Average number of rooms per household.

**Q8. What does Population represent?**

Population of the block group.

**Q9. Why is California Housing suitable for Lasso Regression?**

It contains multiple numerical input features and a continuous target, allowing us to demonstrate regularization and feature selection in a regression problem.

**Q10. Why is Feature Scaling required for this dataset?**

Because its numerical features have different scales, and Lasso penalizes coefficient magnitudes.

# 10. PRACTICAL EXAMINATION – CODING QUESTIONS

**Q1. How do you load a CSV dataset?**

`pd.read_csv("california_housing.csv")`

**Q2. How do you split the dataset?**

`train_test_split(X, y, test_size=0.2)`

**Q3. How do you create a Linear Regression model?**

`LinearRegression()`

**Q4. How do you create a Lasso Regression model?**

`Lasso(alpha=0.1, max_iter=10000)`

**Q5. How do you standardize input features?**

`StandardScaler()`

**Q6. How do you train a Lasso model?**

`model.fit(X_train, y_train)`

**Q7. How do you predict output values?**

`model.predict(X_test)`

**Q8. How do you calculate MSE?**

`mean_squared_error(y_test, y_pred)`

**Q9. How do you calculate R²?**

`r2_score(y_test, y_pred)`

**Q10. How do you extract Lasso coefficients?**

`model.named_steps["regressor"].coef_`

**Q11. How do you identify non-zero coefficients?**

`np.abs(lasso_coef) > 1e-8`

**Q12. How do you identify selected features?**

`X.columns[np.abs(lasso_coef) > 1e-8]`

**Q13. How do you count the selected features?**

`len(selected_features)`

# 11. IMPORTANT FORMULAS

**1. Lasso Regression Objective Function**

L = Σ(Yᵢ - Ŷᵢ)² + αΣ|βⱼ|

**2. Standardization**

Z = (X - μ) / σ

**3. Mean Squared Error**

MSE = (1/n) × Σ(Yᵢ - Ŷᵢ)²

**4. R² Score**

R² = 1 - (SSres / SStot)

**5. L1 Penalty**

L1 Penalty = αΣ|βⱼ|

# 12. IMPORTANT POINTS FOR VIVA

1. Lasso stands for Least Absolute Shrinkage and Selection Operator.
2. Lasso Regression uses L1 regularization.
3. It adds the absolute values of coefficients as a penalty.
4. Alpha controls regularization strength.
5. Lasso can make some regression coefficients exactly zero.
6. It performs automatic feature selection.
7. It helps control overfitting.
8. StandardScaler is important before applying Lasso.
9. Linear Regression is used as the baseline model.
10. Training and testing MSE and R² are compared.
11. A larger alpha does not always improve prediction performance.
12. Lasso can be useful when a dataset contains many input features.